<a href="https://colab.research.google.com/github/hazel00110/grocery_inventory_analysis/blob/main/grocery_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Grocery Inventory - Data Preparation**

---
## 1. Import Libraries & Load Dataset

In [7]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "salahuddinahmedshuvo/grocery-inventory-and-sales-dataset",
    "Grocery_Inventory_and_Sales_Dataset.csv",
)

Using Colab cache for faster access to the 'grocery-inventory-and-sales-dataset' dataset.


In [8]:
df.head(20)

,Product_ID,Product_Name,Catagory,Supplier_ID,Supplier_Name,Stock_Quantity,Reorder_Level,Reorder_Quantity,Unit_Price,Date_Received,Last_Order_Date,Expiration_Date,Warehouse_Location,Sales_Volume,Inventory_Turnover_Rate,Status
0,29-205-1132,Sushi Rice,Grains & Pulses,38-037-1699,Jaxnation,22,72,70,$4.50,8/16/2024,6/29/2024,9/19/2024,48 Del Sol Trail,32,19,Discontinued
1,40-681-9981,Arabica Coffee,Beverages,54-470-2479,Feedmix,45,77,2,$20.00,11/1/2024,5/29/2024,5/8/2024,36 3rd Place,85,1,Discontinued
2,06-955-3428,Black Rice,Grains & Pulses,54-031-2945,Vinder,30,38,83,$6.00,8/3/2024,6/10/2024,9/22/2024,3296 Walton Court,31,34,Backordered
3,71-594-6552,Long Grain Rice,Grains & Pulses,63-492-7603,Brightbean,12,59,62,$1.50,12/8/2024,2/19/2025,4/17/2024,3 Westerfield Crossing,95,99,Active
4,57-437-1828,Plum,Fruits & Vegetables,54-226-4308,Topicstorm,37,30,74,$4.00,7/3/2024,10/11/2024,10/5/2024,15068 Scoville Court,62,25,Backordered
5,21-120-6238,All-Purpose Flour,Grains & Pulses,86-292-4587,Dabjam,55,33,14,$1.75,12/3/2024,5/26/2024,9/5/2024,050 Mcbride Avenue,34,62,Discontinued
6,71-516-1996,Corn Oil,Oils & Fats,04-391-7610,Tagfeed,96,52,16,$2.50,3/18/2024,5/7/2024,6/20/2024,12 Truax Court,67,13,Active
7,39-629-5554,Egg (Goose),Dairy,67-679-4930,Muxo,44,90,17,$2.50,2/3/2025,4/9/2024,2/5/2025,267 International Plaza,21,91,Discontinued
8,66-268-8345,Greek Yogurt,Dairy,32-182-1895,Thoughtstorm,91,84,11,$3.00,12/4/2024,6/2/2024,1/8/2025,550 Clemons Plaza,56,90,Active
9,46-452-9419,Egg (Duck),Dairy,67-137-4215,Wordify,43,10,15,$1.00,11/18/2024,11/14/2024,7/8/2024,55782 Welch Hill,27,69,Active


**Observations from raw data:**
- `Unit_Price` is stored as a string with `$` prefix (e.g. `$4.50`) - needs numeric conversion
- Date columns (`Date_Received`, `Last_Order_Date`, `Expiration_Date`) are strings - need datetime parsing
- `Catagory` has a typo — will be renamed to `category`
- No obvious structural issues with other columns

---
## 2. Data Cleaning

### 2.1 Standardize Column Names

Converting all column names to `snake_case` lowercase for consistency with PostgreSQL conventions downstream.

In [9]:
# Standardize column names: lowercase + snake_case
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('catagory', 'category')  # fix typo
)
print(df.columns.tolist())

['product_id', 'product_name', 'category', 'supplier_id', 'supplier_name', 'stock_quantity', 'reorder_level', 'reorder_quantity', 'unit_price', 'date_received', 'last_order_date', 'expiration_date', 'warehouse_location', 'sales_volume', 'inventory_turnover_rate', 'status']


### 2.2 Fix Data Types

- `unit_price`: strip `$` symbol and convert to float
- Date columns: parse from string to `datetime64`
- Categorical columns: convert to `category` dtype for memory efficiency

In [10]:
# Fix unit_price: remove $ and convert to float
df['unit_price'] = df['unit_price'].str.replace('$', '', regex=False).astype(float)

# Parse date columns
for col in ['date_received', 'last_order_date', 'expiration_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Convert low-cardinality string columns to category dtype
for col in ['category', 'status']:
    df[col] = df[col].astype('category')

print(df.dtypes)

product_id                         object
product_name                       object
category                         category
supplier_id                        object
supplier_name                      object
stock_quantity                      int64
reorder_level                       int64
reorder_quantity                    int64
unit_price                        float64
date_received              datetime64[ns]
last_order_date            datetime64[ns]
expiration_date            datetime64[ns]
warehouse_location                 object
sales_volume                        int64
inventory_turnover_rate             int64
status                           category
dtype: object


---
## 3. Data Quality Check

Checking for:
- Missing values
- Duplicate rows
- Unexpected values in categorical columns

In [11]:
# Check for missing values
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

# Check for duplicate rows
print(f'Duplicates found: {df.duplicated().sum()}')

product_id                 0
product_name               0
category                   1
supplier_id                0
supplier_name              0
stock_quantity             0
reorder_level              0
reorder_quantity           0
unit_price                 0
date_received              0
last_order_date            0
expiration_date            0
warehouse_location         0
sales_volume               0
inventory_turnover_rate    0
status                     0
dtype: int64

Total missing: 1
Duplicates found: 0


In [18]:
df[df['category'].isnull()]

,product_id,product_name,category,supplier_id,supplier_name,stock_quantity,reorder_level,reorder_quantity,unit_price,date_received,last_order_date,expiration_date,warehouse_location,sales_volume,inventory_turnover_rate,status
685,10-378-9729,Cabbage,NaN,83-941-9620,Rooxo,69,21,68,$66.55,12/23/2024,11/26/2024,9/21/2024,2 Butterfield Pass,36,35,Discontinued


In [19]:
df[df['product_name'] == 'Cabbage']

,product_id,product_name,category,supplier_id,supplier_name,stock_quantity,reorder_level,reorder_quantity,unit_price,date_received,last_order_date,expiration_date,warehouse_location,sales_volume,inventory_turnover_rate,status
23,67-025-1245,Cabbage,Fruits & Vegetables,28-575-0716,Chatterbridge,88,46,55,$1.00,4/9/2024,3/6/2024,8/12/2024,51890 Lindbergh Terrace,73,63,Active
501,21-013-3508,Cabbage,Fruits & Vegetables,64-489-8494,Meevee,12,95,49,$1.00,6/9/2024,6/13/2024,11/10/2024,84305 Fair Oaks Plaza,25,23,Backordered
521,79-428-8753,Cabbage,Fruits & Vegetables,29-521-7910,Skivee,62,79,95,$1.00,5/16/2024,7/6/2024,12/30/2024,9 Holmberg Circle,34,12,Active
652,45-380-4627,Cabbage,Fruits & Vegetables,20-201-5639,Zoombox,90,1,40,$0.90,10/20/2024,1/11/2025,2/16/2025,0 Thackeray Point,52,19,Discontinued
685,10-378-9729,Cabbage,NaN,83-941-9620,Rooxo,69,21,68,$66.55,12/23/2024,11/26/2024,9/21/2024,2 Butterfield Pass,36,35,Discontinued
788,75-927-9108,Cabbage,Fruits & Vegetables,27-406-7972,Topdrive,24,32,17,$1.00,9/29/2024,6/17/2024,7/2/2024,43 Washington Street,55,72,Discontinued
806,82-538-4809,Cabbage,Fruits & Vegetables,74-357-2990,Twitternation,83,61,4,$1.00,9/16/2024,11/5/2024,10/24/2024,700 Northland Crossing,35,7,Backordered
987,31-745-6850,Cabbage,Fruits & Vegetables,96-215-2767,Lajo,94,90,12,$0.90,10/3/2024,10/24/2024,11/1/2024,081 Jana Lane,98,71,Active


In [12]:
df['category'].fillna('Fruits & Vegetables', inplace=True)

/tmp/ipykernel_3757/1180315913.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['category'].fillna('Fruits & Vegetables', inplace=True)


In [13]:
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

product_id                 0
product_name               0
category                   0
supplier_id                0
supplier_name              0
stock_quantity             0
reorder_level              0
reorder_quantity           0
unit_price                 0
date_received              0
last_order_date            0
expiration_date            0
warehouse_location         0
sales_volume               0
inventory_turnover_rate    0
status                     0
dtype: int64

Total missing: 0


## 4. Dataset Summary

Final overview of the cleaned dataset before export.

In [16]:
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')

print('\nSummary')
print(df[['stock_quantity','reorder_level','unit_price','sales_volume','inventory_turnover_rate']].describe().round(2))

print('\nDate Range')
print(f'Earliest date_received : {df["date_received"].min().date()}')
print(f'Latest date_received   : {df["date_received"].max().date()}')

Rows: 990 | Columns: 16

Summary
       stock_quantity  reorder_level  unit_price  sales_volume  \
count          990.00         990.00      990.00        990.00   
mean            55.61          51.22        5.92         58.93   
std             26.30          29.10        6.49         23.00   
min             10.00           1.00        0.20         20.00   
25%             33.00          25.25        2.50         39.00   
50%             56.00          53.00        4.22         58.00   
75%             79.00          77.00        7.00         78.00   
max            100.00         100.00       98.43        100.00   

       inventory_turnover_rate  
count                   990.00  
mean                     50.15  
std                      28.80  
min                       1.00  
25%                      25.00  
50%                      50.00  
75%                      74.75  
max                     100.00  

Date Range
Earliest date_received : 2024-02-25
Latest date_received   : 20

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   product_id               990 non-null    object        
 1   product_name             990 non-null    object        
 2   category                 990 non-null    category      
 3   supplier_id              990 non-null    object        
 4   supplier_name            990 non-null    object        
 5   stock_quantity           990 non-null    int64         
 6   reorder_level            990 non-null    int64         
 7   reorder_quantity         990 non-null    int64         
 8   unit_price               990 non-null    float64       
 9   date_received            990 non-null    datetime64[ns]
 10  last_order_date          990 non-null    datetime64[ns]
 11  expiration_date          990 non-null    datetime64[ns]
 12  warehouse_location       990 non-nul

---
## 5. Export

Exporting the cleaned dataset as `grocery_dataset.csv` for ingestion into PostgreSQL.


In [18]:
df.to_csv('grocery_dataset.csv', index=False)
print(f'Exported grocery_dataset.csv — {df.shape[0]} rows × {df.shape[1]} columns')

Exported grocery_dataset.csv — 990 rows × 16 columns
